# 01 — Data Ingestion / Ingesta de Datos

**Project / Proyecto:** `excel-sales-consolidator`  
**Author / Autora:** Cristina  

---

**EN:** This notebook loads 10 raw Excel files with inconsistent formats and diagnoses their structure before normalization. It answers four key questions:
1. What columns exist in each file and how are they named?
2. Are there hidden issues in the column headers (spaces, encoding)?
3. What columns are common across all files?
4. What is the full inventory of unique column names to build the normalization map?

**ES:** Este notebook carga 10 archivos Excel con formatos inconsistentes y diagnostica su estructura antes de la normalización. Responde a cuatro preguntas clave:
1. ¿Qué columnas existen en cada archivo y cómo están nombradas?
2. ¿Existen problemas ocultos en los encabezados (espacios, encoding)?
3. ¿Qué columnas son comunes a todos los archivos?
4. ¿Cuál es el inventario completo de nombres únicos de columna para construir el mapa de normalización?

---

**Input:** `data/raw/source_01.xlsx` → `data/raw/source_10.xlsx`  
**Output:** Diagnostic summary — no files written

## 0 — Setup / Configuración

**EN:** Import required libraries and define the path to the raw datasets.  
**ES:** Importar las librerías necesarias y definir la ruta a los datasets originales.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

print("Libraries loaded successfully")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Resolve root path — works whether notebook runs from /notebooks or project root
ROOT = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

print(f"Project root: {ROOT}")

## 1 — Load Files / Carga de Archivos

**EN:** Load all 10 source files into a list of DataFrames. All columns are read as strings to avoid early type coercion. Column names are stripped of leading/trailing spaces immediately on load.  
**ES:** Carga los 10 archivos fuente en una lista de DataFrames. Todas las columnas se leen como strings para evitar conversiones de tipo prematuras. Los nombres de columna se limpian de espacios al cargar.

In [ ]:
# Generate file names programmatically to avoid manual errors
lista_excel_files = [f"source_{i:02d}.xlsx" for i in range(1, 11)]

lista_dfs = []

for f in lista_excel_files:
    INPUT_FILE = ROOT / "data" / "raw" / f

    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

    df = pd.read_excel(INPUT_FILE, dtype=str)  # Read all columns as string to avoid early type coercion
    df.columns = df.columns.str.strip()        # Strip whitespace from column names immediately on load
    lista_dfs.append(df)

    print(f"✓ {INPUT_FILE.name} — {df.shape[0]} rows × {df.shape[1]} columns")

print(f"\n{len(lista_dfs)} files loaded successfully")

## 2 — Preview / Vista Previa

**EN:** Display the first 5 rows of each file to visually confirm the chaos: inconsistent column names, mixed formats, and missing values.  
**ES:** Muestra las primeras 5 filas de cada archivo para confirmar visualmente el caos: nombres de columna inconsistentes, formatos mixtos y valores nulos.

In [ ]:
for i, df in enumerate(lista_dfs, start=1):
    print(f"\n{'='*50}")
    print(f"source_{i:02d} — {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")
    display(df.head(5))

## 3 — Structure Diagnosis / Diagnóstico de Estructura

**EN:** Inspect shape, column names, and missing values per file to understand the scope of normalization needed.  
**ES:** Inspecciona shape, nombres de columna y valores nulos por archivo para entender el alcance de la normalización necesaria.

In [ ]:
for i, df in enumerate (lista_dfs, start=1):
    print(f"\n{'='*50}")
    print(f"source_{i:02d}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nMissing values:\n{df.isna().sum()}")

## 4 — Column Comparison / Comparación de Columnas

**EN:** Display all column names side by side across all files. This visual comparison is the basis for building the normalization mapping (`COLUMN_MAP`).  
**ES:** Muestra todos los nombres de columna en paralelo para todos los archivos. Esta comparación visual es la base para construir el mapa de normalización (`COLUMN_MAP`).

In [ ]:
# Build a DataFrame with one column per source file, rows = column names (alphabetically sorted)

max_cols = max(len(df.columns) for df in lista_dfs)

mapeo = pd.DataFrame({
    f"source_{i:02d}": sorted(df.columns.tolist(), key=str.lower) + [np.nan] * (max_cols - len(df.columns))
    for i, df in enumerate(lista_dfs, start=1)
})

display(mapeo)


## 5 — Common Columns / Columnas Comunes

**EN:** Identify columns shared across all files. An empty result confirms that no column name is consistent across all sources — which is the exact problem this project solves.  
**ES:** Identifica columnas compartidas por todos los archivos. Un resultado vacío confirma que ningún nombre de columna es consistente en todas las fuentes — que es exactamente el problema que resuelve este proyecto.

In [ ]:

sets_columnas = [set(df.columns) for df in lista_dfs]
comunes = sets_columnas[0].intersection(*sets_columnas[1:])
print("Columnas comunes a todos los archivos:", comunes)

## 6 — Unique Column Inventory / Inventario de Columnas Únicas

**EN:** Extract all unique column names across all files. This inventory is used to manually build the `COLUMN_MAP` in the normalization notebook.  
**ES:** Extrae todos los nombres de columna únicos de todos los archivos. Este inventario se usa para construir manualmente el `COLUMN_MAP` en el notebook de normalización.

In [ ]:
todas_las_columnas = set()
for df in lista_dfs:
    todas_las_columnas.update(df.columns)

columnas_unicas = sorted(todas_las_columnas, key=str.lower)

print(f"Total unique column names across all files: {len(columnas_unicas)}\n")
for col in columnas_unicas:
    print(col)



---

## Summary / Resumen

**EN:**
- 10 files loaded successfully with variable row counts (24–55 rows each)
- No common column names found across all sources — normalization is required
- 53 unique column name variants identified across all files
- Column strip applied on load — no residual whitespace in headers
- → Next step: `02_normalization.ipynb`

**ES:**
- 10 archivos cargados correctamente con número de filas variable (24–55 filas cada uno)
- Ningún nombre de columna común encontrado entre todas las fuentes — se requiere normalización
- 53 variantes únicas de nombres de columna identificadas en todos los archivos
- Strip de columnas aplicado en la carga — sin espacios residuales en encabezados
- → Siguiente paso: `02_normalization.ipynb`